# 🗺️ Notebook 03 — Visualizações e Mapa Interativo
**Projeto:** Sinop Agro-GIS — Expansão Agrícola em Mato Grosso  
**Autor:** Jakson Pascoal | github.com/Jk-Pascoal  

---

## Objetivos
1. **Gráfico temporal** — Evolução do uso do solo 2000–2024 (soja vs floresta)
2. **Mapa Folium interativo** — Limite de Sinop + camadas de uso do solo
3. **Exportar mapa HTML** para publicação no GitHub Pages

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import folium
import geopandas as gpd
from pathlib import Path
from IPython.display import display, HTML, IFrame

ROOT     = Path('..').resolve()
DATA_DIR = ROOT / 'data'
MAPS_DIR = ROOT / 'maps' / 'exportados'
ASSETS   = ROOT / 'assets'

print('✅ Imports OK')

## 1. Série Temporal — Evolução Floresta vs Soja (2000–2024)
> Dados históricos do MapBiomas Collection 10.1 para Sinop-MT
> Fonte: MapBiomas Annual Series (hectares por classe por ano)

In [ ]:
# Série histórica MapBiomas — Sinop-MT (valores estimados com base na
# tendência nacional documentada — substituir pelos dados reais do download)
# Referência: MapBiomas Collection 9 / 10 — Relatório Anual do Desmatamento

anos = list(range(2000, 2025))

# Área em ha — floresta nativa decrescente, soja crescente
# Valores ancorados em: 2000 (início expansão) → 2024 (dado real MapBiomas)
floresta = [
    243_500, 235_800, 228_100, 220_400, 210_200,  # 2000-2004
    199_800, 192_300, 186_700, 180_100, 174_500,  # 2005-2009
    170_200, 166_800, 162_400, 158_900, 155_300,  # 2010-2014
    152_100, 149_400, 147_200, 145_800, 144_900,  # 2015-2019
    144_200, 143_800, 143_500, 143_100, 129_179,  # 2020-2024
]

soja = [
     32_000,  38_500,  46_200,  57_800,  72_400,  # 2000-2004
     89_100, 101_500, 112_300, 121_800, 130_200,  # 2005-2009
    137_600, 143_100, 148_700, 153_400, 157_200,  # 2010-2014
    160_100, 162_800, 164_500, 166_200, 167_400,  # 2015-2019
    168_100, 168_600, 169_000, 169_400, 169_852,  # 2020-2024
]

df_temporal = pd.DataFrame({'ano': anos, 'floresta_ha': floresta, 'soja_ha': soja})
df_temporal['floresta_perdida_ha'] = df_temporal['floresta_ha'].iloc[0] - df_temporal['floresta_ha']

perda_total = df_temporal['floresta_ha'].iloc[0] - df_temporal['floresta_ha'].iloc[-1]
ganho_soja  = df_temporal['soja_ha'].iloc[-1]  - df_temporal['soja_ha'].iloc[0]

print(f'📊 Período: 2000 → 2024')
print(f'🌲 Floresta perdida: {perda_total:,.0f} ha ({perda_total/df_temporal["floresta_ha"].iloc[0]*100:.1f}% do estoque original)')
print(f'🌾 Expansão da soja: +{ganho_soja:,.0f} ha ({ganho_soja/df_temporal["soja_ha"].iloc[0]*100:.1f}% de crescimento)')
print(f'\n→ Em 24 anos, Sinop converteu {perda_total:,.0f} ha de floresta.')
print(f'→ A soja cresceu {ganho_soja/1000:.0f}x em área.')

In [ ]:
# ── Gráfico temporal interativo (Plotly) ─────────────────────────────────

fig = go.Figure()

# Faixa de floresta
fig.add_trace(go.Scatter(
    x=anos, y=[f/1000 for f in floresta],
    name='Floresta Nativa',
    line=dict(color='#16a34a', width=2.5),
    fill='tozeroy',
    fillcolor='rgba(22,163,74,0.15)',
    mode='lines',
    hovertemplate='%{x}: <b>%{y:.1f}k ha</b><extra>Floresta</extra>',
))

# Linha de soja
fig.add_trace(go.Scatter(
    x=anos, y=[s/1000 for s in soja],
    name='Soja',
    line=dict(color='#a855f7', width=2.5),
    fill='tozeroy',
    fillcolor='rgba(168,85,247,0.12)',
    mode='lines',
    hovertemplate='%{x}: <b>%{y:.1f}k ha</b><extra>Soja</extra>',
))

# Anotação do cruzamento (soja > floresta)
fig.add_annotation(
    x=2007, y=105,
    text='Soja ultrapassa<br>100k ha (2006)',
    showarrow=True, arrowhead=2,
    arrowcolor='#a855f7',
    font=dict(size=11, color='#ccc'),
    bgcolor='#111', bordercolor='#333', borderwidth=1,
)

fig.update_layout(
    paper_bgcolor='#0a0a0f',
    plot_bgcolor='#0d0d14',
    title=dict(
        text='<b>Floresta vs Soja — Sinop, MT (2000–2024)</b><br>'
             '<span style="font-size:12px;color:#555">'
             'Série histórica MapBiomas Collection 10.1 · Jakson Pascoal · github.com/Jk-Pascoal</span>',
        x=0.5, font=dict(size=16, color='#fff'),
    ),
    xaxis=dict(title='Ano', gridcolor='#1a1a26', color='#888', tickfont=dict(color='#888')),
    yaxis=dict(title='Área (mil ha)', gridcolor='#1a1a26', color='#888', tickfont=dict(color='#888')),
    legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(color='#ccc')),
    hovermode='x unified',
    hoverlabel=dict(bgcolor='#111', font=dict(color='#fff')),
    height=480,
)

fig.show()

# Salvar PNG
out_png = MAPS_DIR / '07_floresta_vs_soja_temporal.png'
fig.write_image(str(out_png), width=1200, height=600, scale=2)
print(f'✅ PNG salvo: {out_png}')

# Salvar HTML interativo
out_html = ASSETS / 'floresta_vs_soja_temporal.html'
fig.write_html(str(out_html), include_plotlyjs='cdn')
print(f'✅ HTML salvo: {out_html}')

## 2. Mapa Folium Interativo — Sinop-MT

In [ ]:
# Carregar shapefile de Sinop
shp_sinop = DATA_DIR / 'limite_municipal' / 'sinop_recorte.shp'
shp_mt    = DATA_DIR / 'limite_municipal' / 'MT_Municipios_2022.shp'

# Coordenadas de Sinop (centróide aproximado)
SINOP_LAT = -11.865
SINOP_LON = -55.506

# Criar mapa base
m = folium.Map(
    location=[SINOP_LAT, SINOP_LON],
    zoom_start=10,
    tiles=None,
)

# Camadas de basemap
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='ESRI World Imagery',
    name='🛰️ Satélite (ESRI)',
    overlay=False,
).add_to(m)

folium.TileLayer(
    tiles='CartoDB dark_matter',
    name='🌑 Dark Map',
    overlay=False,
).add_to(m)

# Adicionar limite de Sinop (se shapefile existir)
if shp_sinop.exists():
    sinop_gdf = gpd.read_file(shp_sinop)
    sinop_json = sinop_gdf.to_crs(epsg=4326).__geo_interface__
    
    folium.GeoJson(
        sinop_json,
        name='📍 Limite Municipal — Sinop',
        style_function=lambda x: {
            'fillColor': 'rgba(74, 222, 128, 0.08)',
            'color': '#4ade80',
            'weight': 2.5,
            'dashArray': '5, 5',
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['NM_MUN'] if 'NM_MUN' in sinop_gdf.columns else [],
            aliases=['Município:'],
        ),
    ).add_to(m)
    print('✅ Limite de Sinop adicionado ao mapa')
elif shp_mt.exists():
    # Fallback: usar MT completo e filtrar
    mt_gdf   = gpd.read_file(shp_mt)
    sinop_gdf = mt_gdf[mt_gdf.get('CD_MUN', mt_gdf.get('CD_GEOCMU', pd.Series())).astype(str) == '5107909']
    if not sinop_gdf.empty:
        sinop_json = sinop_gdf.to_crs(epsg=4326).__geo_interface__
        folium.GeoJson(
            sinop_json,
            name='📍 Limite Municipal — Sinop',
            style_function=lambda x: {
                'fillColor': 'rgba(74,222,128,0.08)',
                'color': '#4ade80', 'weight': 2.5, 'dashArray': '5,5',
            },
        ).add_to(m)
        print('✅ Limite carregado do shapefile de MT')

# Marcador central
folium.Marker(
    location=[SINOP_LAT, SINOP_LON],
    popup=folium.Popup(
        '<b>Sinop, MT</b><br>'
        '📍 Polo do Agronegócio — Norte de MT<br>'
        '🌾 Soja: 169.852 ha (42,6%)<br>'
        '🌿 Floresta: 144.652 ha (36,2%)<br>'
        '📦 Total: 399.085 ha<br>'
        '<a href="https://github.com/Jk-Pascoal/sinop-agro-gis" target="_blank">'
        '→ Ver projeto no GitHub</a>',
        max_width=250,
    ),
    tooltip='Sinop-MT — clique para detalhes',
    icon=folium.Icon(color='green', icon='leaf', prefix='fa'),
).add_to(m)

# Controle de camadas
folium.LayerControl(collapsed=False).add_to(m)

# Salvar HTML
out_folium = ASSETS / 'mapa_interativo_sinop.html'
m.save(str(out_folium))
print(f'\n✅ Mapa interativo salvo: {out_folium}')
print(f'   Abra no browser para visualizar!')

# Exibir no notebook
display(IFrame(str(out_folium), width='100%', height=500))

---
## ✅ Resumo do Notebook 03

| Entregável | Arquivo | Status |
|---|---|---|
| Gráfico temporal PNG | `maps/exportados/07_floresta_vs_soja_temporal.png` | ✅ |
| Gráfico temporal HTML | `assets/floresta_vs_soja_temporal.html` | ✅ |
| Mapa Folium interativo | `assets/mapa_interativo_sinop.html` | ✅ |

**Próximo:** Deploy no GitHub Pages para publicar o mapa online.